# NLP com Deep Learning

(e séries temporais)

Nesse notebook vamos experimentar alguns modelos de NLP usando o Keras:

- `NER`: Reconhecimento de Entidades Nomeadas (do inglês Named Entity Recognition), utilizado como extração de dados relevantes de um texto. A arquitetura do Keras é a de `many-to-many`, realizando uma tarefa de `classificação de token`.
- `Classificação de texto`: utilizado para catalogar textos em determinadas categorias. A arquitetura é de `many-to-one`, onde o texto inteiro é classificado em uma categoria.


E também vamos ver como fazer previsão de séries temporais!

## NER

Vamos utilizar uma base de dados brasileira: [LeNER-Br](https://raw.githubusercontent.com/peluz/lener-br/refs/heads/master/leNER-Br/train/train.conll), contendo textos da área jurídica do Brasil .

In [ ]:
!wget -q https://raw.githubusercontent.com/peluz/lener-br/refs/heads/master/leNER-Br/train/train.conll
!wget -q https://raw.githubusercontent.com/peluz/lener-br/refs/heads/master/leNER-Br/test/test.conll
!wget -q https://raw.githubusercontent.com/peluz/lener-br/refs/heads/master/leNER-Br/dev/dev.conll

Dados para treino de modelos NER geralmente são compartilhadas em formato CoNLL com sistema de tagueamento IOB. Vamos ver as primeiras linhas do arquivo:

In [ ]:
!head -20 dev.conll

Cada linha possui uma palavra e a tag associada (veja que os dados não são limpos). Tags iniciadas com `B-` são o início de uma entidade, enquanto que tags com `I-` são continuação de entidades. Tags `O` significam outro (sem classe). Cada frase é separada por uma linha em branco.

In [ ]:
def carrega_conll(arquivo):
    """Carrega arquivo CoNLL

    Lê o arquivo no formato CoNLL esperando duas colunas: palavras e tags.
    Cria listas de palavras e tags para cada frase.

    Parameters
    ----------
    arquivo : str
        Caminho para o arquivo (train/dev/test).

    Returns
    -------
    list:
        Lista de dicionários com palavras e tags de cada frase.

    """
    dados = []
    tokens = []
    tags = []
    with open(arquivo, 'r') as f:
        for linha in f:
            # Separa cada frase
            if linha == '\n':
              if len(tokens) > 0 and len(tokens) == len(tags):
                dados.append({'tokens': tokens, 'tags': tags})

              tokens = []
              tags = []
              continue

            token, tag = linha.split()
            tokens.append(token)
            tags.append(tag)

    return dados

In [ ]:
train = carrega_conll("train.conll")
dev = carrega_conll('dev.conll')
test = carrega_conll("test.conll")

In [ ]:
print(f"Tamanho da base de treino: {len(train)}")
print(f"Tamanho da base de validação: {len(dev)}")
print(f"Tamanho da base de teste: {len(test)}")

In [ ]:
train[0]

Vamos preparar os dados para uso em redes neurais:

- Criar um índice para as palavras
- Criar um índice para as tags
- Padronizar o tamanho das frases

In [ ]:
todas_palavras = []
todas_tags = []

for frase in train:
  todas_palavras.extend(frase['tokens'])
  todas_tags.extend(frase['tags'])

palavras_unicas = sorted(set(todas_palavras))
tags_unicas = sorted(set(todas_tags), key=lambda x: "0" if x == "O" else x.split("-")[-1])

print(f"Número de palavras únicas: {len(palavras_unicas)}")
print(f"Número de tags únicas: {len(tags_unicas)}")

In [ ]:
tags_unicas

In [ ]:
# Vamos reservar o índice 1 para um token especial de PAD
# Vamos reservar o índice 2 para OOV (out of vocabulary)
palavra_idx = {w: i + 2 for i, w in enumerate(palavras_unicas)}
palavra_idx['<PAD>'] = 0
palavra_idx['<OOV>'] = 1
oov_idx = palavra_idx['<OOV>']
tag_idx = {t: i for i, t in enumerate(tags_unicas)}
o_tag = tag_idx["O"]

In [ ]:
idx_palavra = {i: w for w, i in palavra_idx.items()}
idx_tag = {i: w for w, i in tag_idx.items()}

In [ ]:
tag_idx["O"]

In [ ]:
idx_tag[2]

In [ ]:
palavra_idx["PÚBLICO"]

In [ ]:
palavra_idx["público"]

In [ ]:
idx_palavra[12000]

Transformando texto em números e padronizando tamanho.

In [ ]:
from keras.utils import pad_sequences, to_categorical
import numpy as np

In [ ]:
frases_lista_train = [[palavra_idx.get(x, oov_idx) for x in frase['tokens']] for frase in train]
frases_lista_dev = [[palavra_idx.get(x, oov_idx) for x in frase['tokens']] for frase in dev]
frases_lista_test = [[palavra_idx.get(x, oov_idx) for x in frase['tokens']] for frase in test]

tags_lista_train = [[tag_idx.get(x, o_tag) for x in frase['tags']] for frase in train]
tags_lista_dev = [[tag_idx.get(x, o_tag) for x in frase['tags']] for frase in dev]
tags_lista_test = [[tag_idx.get(x, o_tag) for x in frase['tags']] for frase in test]


frases_lista_train[0]

Colocamos a tag <PAD> no início do text (também pode ser feito no final). Para colocar tags de PAD no inicio, usamos `padding="pre"`, para colocar no final, usamos `padding="post"`.

In [ ]:
MAXLEN = 600

# Padroniza tamanho das frases
Xtrain_pad = pad_sequences(frases_lista_train, maxlen=MAXLEN, padding='pre', value=0)
Xdev_pad = pad_sequences(frases_lista_dev, maxlen=MAXLEN, padding='pre', value=0)
Xtest_pad = pad_sequences(frases_lista_test, maxlen=MAXLEN, padding='pre', value=0)

# Padroniza tamanho das tags
Ytrain_pad = pad_sequences(tags_lista_train, maxlen=MAXLEN, padding='pre', value=o_tag)
Ydev_pad = pad_sequences(tags_lista_dev, maxlen=MAXLEN, padding='pre', value=o_tag)
Ytest_pad = pad_sequences(tags_lista_test, maxlen=MAXLEN, padding='pre', value=o_tag)

In [ ]:
# Transforma uma tag em um vetor para classificação -- 1 -> [1 , 0, 0 , ..]
Ytrain_cat = np.array([to_categorical(x, num_classes=len(tag_idx)) for x in Ytrain_pad])
Ydev_cat = np.array([to_categorical(x, num_classes=len(tag_idx)) for x in Ydev_pad])
Ytest_cat = np.array([to_categorical(x, num_classes=len(tag_idx)) for x in Ytest_pad])

In [ ]:
# Tamanho máximo entre as frases
max([len(x) for x in frases_lista_train])

Vamos criar uma rede neural usando camadas `GRU` para classificar cada token/palavra das frases. Para modelos `many-to-many` como NER, aonde cada token vai ser classificado, utilizamos a camada auxiliar [TimeDistributed](https://keras.io/api/layers/recurrent_layers/time_distributed/).

Essa camada irá aplicar a camada desejada (per exemplo, a `Dense`) em cada entrada, colocando no nosso modelo um classificador para cada palavra.

In [ ]:
from keras.layers import Embedding, Dense, Input, GRU, Bidirectional, TimeDistributed
from keras.models import Sequential

In [ ]:
model = Sequential(name="NER_model")

model.add(Input(shape=(MAXLEN,), dtype="int32"))

# Cria embeddings com dimensão 200
model.add(Embedding(input_dim=len(palavra_idx), output_dim=200))

# Adiciona camada bidirecional de GRU
model.add(Bidirectional(GRU(units=100, return_sequences=True, dropout=0.15)))

# Mais uma cama GRU
model.add(GRU(50, return_sequences=True))

# Classificadores
model.add(TimeDistributed(Dense(len(tag_idx), activation='softmax')))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
EPOCHS = 5
BATCH_SIZE=128
model.fit(Xtrain_pad,Ytrain_cat, validation_data=(Xdev_pad, Ydev_cat), epochs=EPOCHS, batch_size=BATCH_SIZE)

In [ ]:
model.evaluate(Xtest_pad, Ytest_cat)

Vamos fazer uma análise mais detalhada do modelo utilizando o `scikitlearn`.

In [ ]:
pred = model.predict(Xtest_pad[:1])
pred

O modelo cria um vetor com as probabilidades de cada tag para cada palavra do input. Vamos selectionar a tag com maior probabilidade para fazer nossa predição.

In [ ]:
# Exemplo
pred_cat = [idx_tag[np.argmax(x)] for x in pred[0]]
pred_cat

Para a base de treino:

In [ ]:
preds = model.predict(Xtest_pad)

In [ ]:
# Pega índice de maior probabilidade e transforma índice em tag com o dicionário
preds_cat = [
    [idx_tag[np.argmax(x)] for x in pred] for pred in preds
]

len(preds_cat[0]), len(Ytest_pad[0])

Para utilizar o `scikitlearn`, vamos transformar as predições em uma lista única, assim podemos verificar o desempenho para cada tag.

In [ ]:
y_pred = [x for pred in preds_cat for x in pred]
y_true = [idx_tag[x] for tags in Ytest_pad for x in tags]

In [ ]:
len(y_true), len(y_pred)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
confusion_matrix(y_true, y_pred)

In [ ]:
print(classification_report(y_true, y_pred, zero_division=0))

Sua análise: O modelo BiGRU apresenta bom desempenho para a tag O, mas sofre com o desbalanceamento das classes, afetando entidades raras. A arquitetura está correta, porém pode ser aprimorada com CRF, embeddings pré-treinados e métricas por entidade (F1).

### **Desafio**: Crie o mesmo modelo, mas utilizando um embedding pretreinado! Veja como utilizar [aqui](https://keras.io/examples/nlp/pretrained_word_embeddings/), mas precisa de um modelo para o [português](http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc).

## Classificação de texto

Vamos ao exemplo clássico: análise de sentimento em críticas de filmes!

In [ ]:
from keras.datasets import imdb
from keras.layers import LSTM
from keras.models import Model

In [ ]:
# Limita a top max_features palavras
max_features = 20000

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=max_features
)
print(len(x_train), "Training sequences")
print(len(x_test), "Test sequences")

In [ ]:
len(x_train[0]), len(x_train[1])

In [ ]:
maxlen = 200

x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

Criando o modelo

In [ ]:
inputs = Input(shape=(None,), dtype="int32")

x = Embedding(max_features, 128)(inputs)

x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.1))(x)

# Camada LSTM sem retornar as sequencias
x = LSTM(64, dropout=0.1)(x)

# Classificador binário
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Usamos o validation_split para criar base de validação (0.15)
model.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.15)


In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
preds = model.predict(x_test)

threshold = 0.5
preds = [1 if p > threshold else 0 for p in preds]

print(classification_report(y_test, preds, zero_division=0))

In [ ]:
confusion_matrix(y_test, preds)

Sua análise: O modelo BiLSTM captura bem dependências contextuais e apresenta bom desempenho na classificação binária de sentimentos. A arquitetura é adequada, mas pode ser aprimorada com regularização adicional, embeddings pré-treinados ou ajuste de hiperparâmetros para melhorar a generalização.

Para pensar: O que acontece com as classificações se mudamos os valores do `threshold`?

### **Desafio**: Melhore a acurácia do modelo modificando as camadas.

## Séries Temporais

Nesse exemplo, vemos como prever séreis temporais de clima com RNNs!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from keras.utils import get_file
from zipfile import ZipFile

In [ ]:
# Carregando dados
uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
zip_file = ZipFile(zip_path)
zip_file.extractall()
csv_path = "jena_climate_2009_2016.csv"

df = pd.read_csv(csv_path)

In [ ]:
df.head()

Visualizando as séries temporais.

In [ ]:
titles = [
    "Pressure",
    "Temperature",
    "Temperature in Kelvin",
    "Temperature (dew point)",
    "Relative Humidity",
    "Saturation vapor pressure",
    "Vapor pressure",
    "Vapor pressure deficit",
    "Specific humidity",
    "Water vapor concentration",
    "Airtight",
    "Wind speed",
    "Maximum wind speed",
    "Wind direction in degrees",
]

feature_keys = [
    "p (mbar)",
    "T (degC)",
    "Tpot (K)",
    "Tdew (degC)",
    "rh (%)",
    "VPmax (mbar)",
    "VPact (mbar)",
    "VPdef (mbar)",
    "sh (g/kg)",
    "H2OC (mmol/mol)",
    "rho (g/m**3)",
    "wv (m/s)",
    "max. wv (m/s)",
    "wd (deg)",
]

colors = [
    "blue",
    "orange",
    "green",
    "red",
    "purple",
    "brown",
    "pink",
    "gray",
    "olive",
    "cyan",
]

date_time_key = "Date Time"


def show_raw_visualization(data):
    time_data = data[date_time_key]
    fig, axes = plt.subplots(
        nrows=7, ncols=2, figsize=(15, 20), dpi=80, facecolor="w", edgecolor="k"
    )
    for i in range(len(feature_keys)):
        key = feature_keys[i]
        c = colors[i % (len(colors))]
        t_data = data[key]
        t_data.index = time_data
        t_data.head()
        ax = t_data.plot(
            ax=axes[i // 2, i % 2],
            color=c,
            title="{} - {}".format(titles[i], key),
            rot=25,
        )
        ax.legend([titles[i]])
    plt.tight_layout()


show_raw_visualization(df)


Normalizando os dados

In [ ]:
split_fraction = 0.715
train_split = int(split_fraction * int(df.shape[0]))
step = 6

past = 720
future = 72
learning_rate = 0.001
batch_size = 256
epochs = 10


def normalize(data, train_split):
    data_mean = data[:train_split].mean(axis=0)
    data_std = data[:train_split].std(axis=0)
    return (data - data_mean) / data_std

In [ ]:
print(
    "Os parâmetros para o modelo são:",
    ", ".join([titles[i] for i in [0, 1, 5, 7, 8, 10, 11]]),
)
selected_features = [feature_keys[i] for i in [0, 1, 5, 7, 8, 10, 11]]
features = df[selected_features]
features.index = df[date_time_key]
features.head()

In [ ]:
features = normalize(features.values, train_split)
features = pd.DataFrame(features)
features.head()

In [ ]:
train_data = features.loc[0 : train_split - 1]
val_data = features.loc[train_split:]

Separa em base de treino e teste

In [ ]:
start = past + future
end = start + train_split

x_train = train_data[[i for i in range(7)]].values
y_train = features.iloc[start:end][[1]]

sequence_length = int(past / step)

In [ ]:
from keras.preprocessing import timeseries_dataset_from_array

dataset_train = timeseries_dataset_from_array(
    x_train,
    y_train,
    sequence_length=sequence_length,
    sampling_rate=step, # Transforma em uma observação por hora
    batch_size=batch_size,
)

A base de teste deve ser sempre o futuro!

In [ ]:
x_end = len(val_data) - past - future

label_start = train_split + past + future

x_val = val_data.iloc[:x_end][[i for i in range(7)]].values
y_val = features.iloc[label_start:][[1]]

dataset_val = timeseries_dataset_from_array(
    x_val,
    y_val,
    sequence_length=sequence_length,
    sampling_rate=step,
    batch_size=batch_size,
)


for batch in dataset_train.take(1):
    inputs, targets = batch

print("Input shape:", inputs.numpy().shape)
print("Target shape:", targets.numpy().shape)

Cria o modelo

In [ ]:
inputs = Input(shape=(inputs.shape[1], inputs.shape[2]))

# Camadas RNN - vamos usar o LSTM
lstm_out = LSTM(32, return_sequences=True, dropout=0.1)(inputs)
lstm_out = LSTM(16)(lstm_out)

# Regressor
outputs = Dense(1)(lstm_out)

model = Model(inputs=inputs, outputs=outputs)

# Agora utilizamos a função de perda MSE
model.compile(optimizer="adam", loss="mse")

model.summary()


In [ ]:
history = model.fit(
    dataset_train,
    epochs=epochs,
    validation_data=dataset_val,
)


In [ ]:
def visualize_loss(history, title):
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(len(loss))
    plt.figure()
    plt.plot(epochs, loss, "b", label="Training loss")
    plt.plot(epochs, val_loss, "r", label="Validation loss")
    plt.title(title)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

In [ ]:
visualize_loss(history, "Training and Validation Loss")

In [ ]:
def show_plot(plot_data, delta, title):
    labels = ["History", "True Future", "Model Prediction"]
    marker = [".-", "rx", "go"]
    time_steps = list(range(-(plot_data[0].shape[0]), 0))
    if delta:
        future = delta
    else:
        future = 0

    plt.title(title)
    for i, val in enumerate(plot_data):
        if i:
            plt.plot(future, plot_data[i], marker[i], markersize=10, label=labels[i])
        else:
            plt.plot(time_steps, plot_data[i].flatten(), marker[i], label=labels[i])
    plt.legend()
    plt.xlim([time_steps[0], (future + 5) * 2])
    plt.xlabel("Time-Step")
    plt.show()
    return

In [ ]:
for x, y in dataset_val.take(5):
    show_plot(
        [x[0][:, 1].numpy(), y[0].numpy(), model.predict(x)[0]],
        12,
        "Single Step Prediction",
    )

Sua análise:O modelo LSTM consegue capturar dependências temporais relevantes para previsão de temperatura, apresentando convergência estável entre treino e validação. A normalização e a criação correta das janelas temporais contribuem para o bom desempenho, embora arquiteturas mais profundas ou ajustes de hiperparâmetros possam melhorar a precisão.

### **Desafio**: Crie um modelo para prever o preço de ações.

Referência
Fischer, T., & Krauss, C. (2018).
Deep learning with long short-term memory networks for financial market predictions.
European Journal of Operational Research, 270(2), 654–669.
https://doi.org/10.1016/j.ejor.2017.11.054

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Baixa dados da PETR4 (B3)
df = yf.download("PETR4.SA", start="2015-01-01", end="2024-01-01")

# Usaremos o preço de fechamento
data = df[["Close"]]
data.head()

In [ ]:
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

In [ ]:
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [ ]:
window_size = 60
X, y = create_sequences(data_scaled, window_size)

# Separação treino/teste
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense

n_features = 1

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(window_size, n_features)),
    LSTM(50),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)


In [ ]:
predictions = model.predict(X_test)

predictions = scaler.inverse_transform(predictions)
y_test_real = scaler.inverse_transform(y_test)
